In [35]:
# Cell 1 - Import library

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN

import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print("Libraries loaded.")

Libraries loaded.


In [36]:
# Cell 2 - Konfigurasi utama

# ============================================================
# INPUT FILE CONFIG
# ============================================================


CHECK_FILE_PATH = Path(
    "/media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008/GC25_Z080/Dataset Development/Bungkuk/Bustan/2.csv"
)


# ============================================================
# DBSCAN FIXED PARAMETER
# ============================================================

EPS = 0.20
MIN_SAMPLES = 15
MIN_CLUSTER_POINTS = 20

# ============================================================
# DBSCAN FEATURES
# ============================================================

# Fitur yang dipakai DBSCAN
DBSCAN_FEATURES = ["X_corr", "Y_corr", "Z_level"]

# Kolom untuk mendeteksi titik corrected all-zero
# Jangan pakai Z_level untuk all-zero check karena Z_level bisa positif akibat floor_offset.
CORR_ZERO_CHECK_COLUMNS = ["X_corr", "Y_corr", "Z_corr"]

# Kolom raw untuk mendeteksi titik raw all-zero
RAW_ZERO_CHECK_COLUMNS = ["X", "Y", "Z"]

# Kolom yang harus finite
FINITE_CHECK_COLUMNS = ["X_corr", "Y_corr", "Z_corr", "Z_level"]

# ============================================================
# CLEANING CONFIG
# ============================================================

REMOVE_NON_FINITE = True
REMOVE_ALL_ZERO_COORDS = True
DEGENERATE_EPS = 1e-6

# ============================================================
# VISUAL AXIS CONFIG
# ============================================================

ROI_X_MIN, ROI_X_MAX = 0.0, 3.0
ROI_Y_MIN, ROI_Y_MAX = -1.5, 1.5
ROI_Z_MIN, ROI_Z_MAX = 0.0, 2.0

MAX_POINTS_PLOT = 15000

# ============================================================
# VISUAL MODES
# ============================================================

VISUAL_MODES = [
    "labels",
    "main_only",
    "noise_vs_cluster",
    "valid_clusters_only",
    "raw_cleaned",
]

DEFAULT_VISUAL_MODE = "labels"

print("===== VISUAL VALIDATION CONFIG =====")
print(f"File path          : {CHECK_FILE_PATH}")
print(f"File exists        : {CHECK_FILE_PATH.exists()}")
print(f"EPS                : {EPS}")
print(f"MIN_SAMPLES        : {MIN_SAMPLES}")
print(f"MIN_CLUSTER_POINTS : {MIN_CLUSTER_POINTS}")
print(f"DBSCAN_FEATURES    : {DBSCAN_FEATURES}")
print(f"Visual modes       : {VISUAL_MODES}")

===== VISUAL VALIDATION CONFIG =====
File path          : /media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008/GC25_Z080/Dataset Development/Bungkuk/Bustan/2.csv
File exists        : True
EPS                : 0.2
MIN_SAMPLES        : 15
MIN_CLUSTER_POINTS : 20
DBSCAN_FEATURES    : ['X_corr', 'Y_corr', 'Z_level']
Visual modes       : ['labels', 'main_only', 'noise_vs_cluster', 'valid_clusters_only', 'raw_cleaned']


In [37]:
# Cell 3 - Load file dan validasi kolom

REQUIRED_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

if not CHECK_FILE_PATH.exists():
    raise FileNotFoundError(f"File not found: {CHECK_FILE_PATH}")

df = pd.read_csv(CHECK_FILE_PATH)

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

frame_ids = sorted(df["frame_id"].dropna().unique().tolist())

print("===== FILE LOADED =====")
print(f"Shape       : {df.shape}")
print(f"Frames      : {len(frame_ids)}")
print(f"First frame : {frame_ids[0] if frame_ids else None}")
print(f"Last frame  : {frame_ids[-1] if frame_ids else None}")

display(df.head())

===== FILE LOADED =====
Shape       : (117478, 10)
Frames      : 66
First frame : 0
Last frame  : 65


,frame_id,Timestamp,X,Y,Z,X_corr,Y_corr,Z_corr,Z_level,Reflectivity
0,0,226807882880,0.0,0.0,0.0,0.0,0.0,0.0,0.891725,50.0
1,0,226808362880,0.0,0.0,0.0,0.0,0.0,0.0,0.891725,50.0
2,0,226808362880,0.0,0.0,0.0,0.0,0.0,0.0,0.891725,50.0
3,0,226808362880,0.0,0.0,0.0,0.0,0.0,0.0,0.891725,0.0
4,0,226808362880,0.0,0.0,0.0,0.0,0.0,0.0,0.891725,50.0


In [38]:
# Cell 4 - Fungsi cleaning, DBSCAN, dan metrik frame

def clean_degenerate_points(frame_df: pd.DataFrame):
    """
    Cleaning ringan sebelum DBSCAN.

    Remove:
    1. NaN / Inf pada X_corr, Y_corr, Z_corr, Z_level
    2. Corrected all-zero: X_corr≈0, Y_corr≈0, Z_corr≈0
    3. Raw all-zero: X≈0, Y≈0, Z≈0

    DBSCAN tetap memakai X_corr, Y_corr, Z_level.
    """
    n_before = len(frame_df)

    if n_before == 0:
        return frame_df.copy(), {
            "n_points_before_cleaning": 0,
            "n_non_finite_removed": 0,
            "n_corr_all_zero_removed": 0,
            "n_raw_all_zero_removed": 0,
            "n_all_zero_removed": 0,
            "n_points_after_cleaning": 0,
            "cleaning_removed_ratio": np.nan,
        }

    out_df = frame_df.copy()

    check_cols = list(set(
        DBSCAN_FEATURES
        + CORR_ZERO_CHECK_COLUMNS
        + RAW_ZERO_CHECK_COLUMNS
        + FINITE_CHECK_COLUMNS
    ))

    for col in check_cols:
        out_df[col] = pd.to_numeric(out_df[col], errors="coerce")

    finite_values = out_df[FINITE_CHECK_COLUMNS].to_numpy(dtype=float)

    if REMOVE_NON_FINITE:
        finite_mask = np.isfinite(finite_values).all(axis=1)
    else:
        finite_mask = np.ones(len(out_df), dtype=bool)

    corr_values = out_df[CORR_ZERO_CHECK_COLUMNS].to_numpy(dtype=float)
    corr_all_zero_mask = (
        (np.abs(corr_values[:, 0]) < DEGENERATE_EPS) &
        (np.abs(corr_values[:, 1]) < DEGENERATE_EPS) &
        (np.abs(corr_values[:, 2]) < DEGENERATE_EPS)
    )

    raw_values = out_df[RAW_ZERO_CHECK_COLUMNS].to_numpy(dtype=float)
    raw_all_zero_mask = (
        (np.abs(raw_values[:, 0]) < DEGENERATE_EPS) &
        (np.abs(raw_values[:, 1]) < DEGENERATE_EPS) &
        (np.abs(raw_values[:, 2]) < DEGENERATE_EPS)
    )

    if not REMOVE_ALL_ZERO_COORDS:
        corr_all_zero_mask = np.zeros(len(out_df), dtype=bool)
        raw_all_zero_mask = np.zeros(len(out_df), dtype=bool)

    non_finite_mask = ~finite_mask
    zero_union_mask = corr_all_zero_mask | raw_all_zero_mask

    keep_mask = finite_mask & (~zero_union_mask)

    n_non_finite_removed = int(non_finite_mask.sum())
    n_corr_all_zero_removed = int((finite_mask & corr_all_zero_mask).sum())
    n_raw_all_zero_removed = int((finite_mask & raw_all_zero_mask).sum())
    n_all_zero_removed = int((finite_mask & zero_union_mask).sum())

    cleaned_df = out_df.loc[keep_mask].copy()
    n_after = len(cleaned_df)

    cleaning_metrics = {
        "n_points_before_cleaning": int(n_before),
        "n_non_finite_removed": n_non_finite_removed,
        "n_corr_all_zero_removed": n_corr_all_zero_removed,
        "n_raw_all_zero_removed": n_raw_all_zero_removed,
        "n_all_zero_removed": n_all_zero_removed,
        "n_points_after_cleaning": int(n_after),
        "cleaning_removed_ratio": (n_before - n_after) / n_before if n_before > 0 else np.nan,
    }

    return cleaned_df, cleaning_metrics


def safe_range(series):
    if series is None or len(series) == 0:
        return np.nan, np.nan, np.nan

    arr = np.asarray(series)
    if len(arr) == 0:
        return np.nan, np.nan, np.nan

    vmin = float(np.min(arr))
    vmax = float(np.max(arr))
    return vmin, vmax, vmax - vmin


def apply_dbscan_one_frame(frame_df: pd.DataFrame):
    """
    Cleaning + DBSCAN untuk satu frame.
    Output:
    - dataframe dengan dbscan_label, cluster flags
    - metrics dictionary
    """
    n_input = len(frame_df)

    cleaned_df, cleaning_metrics = clean_degenerate_points(frame_df)

    if len(cleaned_df) == 0:
        labeled_df = cleaned_df.copy()
        labeled_df["dbscan_label"] = []
        labeled_df["is_noise"] = []
        labeled_df["is_valid_cluster"] = []
        labeled_df["is_main_cluster"] = []

        metrics = {
            "status": "empty_after_cleaning",
            "n_points_input": n_input,
            **cleaning_metrics,
            "cluster_count_total": 0,
            "cluster_count_valid": 0,
            "noise_points": 0,
            "noise_ratio": np.nan,
            "main_cluster_label": np.nan,
            "main_cluster_points": 0,
            "main_cluster_ratio_total": 0.0,
            "main_cluster_x_range": np.nan,
            "main_cluster_y_range": np.nan,
            "main_cluster_z_range": np.nan,
        }
        return labeled_df, metrics

    if len(cleaned_df) < MIN_SAMPLES:
        labeled_df = cleaned_df.copy()
        labeled_df["dbscan_label"] = -1
        labeled_df["is_noise"] = True
        labeled_df["is_valid_cluster"] = False
        labeled_df["is_main_cluster"] = False

        metrics = {
            "status": "too_few_points",
            "n_points_input": n_input,
            **cleaning_metrics,
            "cluster_count_total": 0,
            "cluster_count_valid": 0,
            "noise_points": len(cleaned_df),
            "noise_ratio": 1.0,
            "main_cluster_label": np.nan,
            "main_cluster_points": 0,
            "main_cluster_ratio_total": 0.0,
            "main_cluster_x_range": np.nan,
            "main_cluster_y_range": np.nan,
            "main_cluster_z_range": np.nan,
        }
        return labeled_df, metrics

    X_feat = cleaned_df[DBSCAN_FEATURES].to_numpy(dtype=float)

    labels = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit_predict(X_feat)

    labeled_df = cleaned_df.copy()
    labeled_df["dbscan_label"] = labels
    labeled_df["is_noise"] = labeled_df["dbscan_label"] == -1

    cluster_labels = sorted([int(lab) for lab in np.unique(labels) if lab != -1])
    cluster_sizes = {
        lab: int((labels == lab).sum())
        for lab in cluster_labels
    }

    valid_cluster_labels = [
        lab for lab, size in cluster_sizes.items()
        if size >= MIN_CLUSTER_POINTS
    ]

    labeled_df["is_valid_cluster"] = labeled_df["dbscan_label"].isin(valid_cluster_labels)

    if len(valid_cluster_labels) > 0:
        main_cluster_label = max(valid_cluster_labels, key=lambda lab: cluster_sizes[lab])
    else:
        main_cluster_label = np.nan

    labeled_df["is_main_cluster"] = labeled_df["dbscan_label"] == main_cluster_label

    n_total = len(labeled_df)
    noise_points = int((labeled_df["dbscan_label"] == -1).sum())
    main_cluster_points = int(labeled_df["is_main_cluster"].sum())

    if main_cluster_points > 0:
        main_df = labeled_df[labeled_df["is_main_cluster"]].copy()
        _, _, x_range = safe_range(main_df["X_corr"])
        _, _, y_range = safe_range(main_df["Y_corr"])
        _, _, z_range = safe_range(main_df["Z_level"])
    else:
        x_range = y_range = z_range = np.nan

    if len(valid_cluster_labels) == 0:
        status = "no_valid_cluster"
    else:
        status = "success"

    metrics = {
        "status": status,
        "n_points_input": n_input,
        **cleaning_metrics,
        "cluster_count_total": len(cluster_labels),
        "cluster_count_valid": len(valid_cluster_labels),
        "noise_points": noise_points,
        "noise_ratio": noise_points / n_total if n_total > 0 else np.nan,
        "main_cluster_label": main_cluster_label,
        "main_cluster_points": main_cluster_points,
        "main_cluster_ratio_total": main_cluster_points / n_total if n_total > 0 else np.nan,
        "main_cluster_x_range": x_range,
        "main_cluster_y_range": y_range,
        "main_cluster_z_range": z_range,
    }

    return labeled_df, metrics

In [39]:
# Cell 5 - Proses seluruh frame dalam satu file

frame_results = {}
frame_metrics = []

print("===== PROCESSING FRAMES FOR VISUAL VALIDATION =====")
print(f"Total frames: {len(frame_ids)}")
print(f"EPS={EPS}, MIN_SAMPLES={MIN_SAMPLES}, MIN_CLUSTER_POINTS={MIN_CLUSTER_POINTS}")

for idx, frame_id in enumerate(frame_ids, start=1):
    frame_df = df[df["frame_id"] == frame_id].copy()

    labeled_df, metrics = apply_dbscan_one_frame(frame_df)

    metrics["frame_id"] = frame_id
    metrics["frame_index"] = idx - 1

    frame_results[frame_id] = {
        "labeled_df": labeled_df,
        "metrics": metrics,
    }

    frame_metrics.append(metrics)

    if idx == 1 or idx % 10 == 0 or idx == len(frame_ids):
        print(
            f"[{idx}/{len(frame_ids)}] frame={frame_id} | "
            f"status={metrics['status']} | "
            f"cleaned={metrics['n_points_after_cleaning']} | "
            f"clusters={metrics['cluster_count_valid']} | "
            f"noise={metrics['noise_ratio']:.4f} | "
            f"main_ratio={metrics['main_cluster_ratio_total']:.4f}"
        )

frame_metrics_df = pd.DataFrame(frame_metrics)

print("===== DONE =====")
display(frame_metrics_df.head())

print("===== FRAME STATUS SUMMARY =====")
display(
    frame_metrics_df["status"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "status", "status": "count"})
)

===== PROCESSING FRAMES FOR VISUAL VALIDATION =====
Total frames: 66
EPS=0.2, MIN_SAMPLES=15, MIN_CLUSTER_POINTS=20
[1/66] frame=0 | status=success | cleaned=973 | clusters=1 | noise=0.0000 | main_ratio=1.0000
[10/66] frame=9 | status=success | cleaned=1007 | clusters=1 | noise=0.0000 | main_ratio=1.0000
[20/66] frame=19 | status=success | cleaned=1031 | clusters=1 | noise=0.0000 | main_ratio=1.0000
[30/66] frame=29 | status=success | cleaned=837 | clusters=1 | noise=0.0000 | main_ratio=1.0000
[40/66] frame=39 | status=success | cleaned=1579 | clusters=1 | noise=0.0000 | main_ratio=1.0000
[50/66] frame=49 | status=success | cleaned=1535 | clusters=1 | noise=0.0000 | main_ratio=1.0000
[60/66] frame=59 | status=success | cleaned=1024 | clusters=1 | noise=0.0000 | main_ratio=1.0000
[66/66] frame=65 | status=success | cleaned=1038 | clusters=1 | noise=0.0000 | main_ratio=1.0000
===== DONE =====


,status,n_points_input,n_points_before_cleaning,n_non_finite_removed,n_corr_all_zero_removed,n_raw_all_zero_removed,n_all_zero_removed,n_points_after_cleaning,cleaning_removed_ratio,cluster_count_total,...,noise_points,noise_ratio,main_cluster_label,main_cluster_points,main_cluster_ratio_total,main_cluster_x_range,main_cluster_y_range,main_cluster_z_range,frame_id,frame_index
0,success,1478,1478,0,505,505,505,973,0.341678,1,...,0,0.0,0,973,1.0,0.202299,0.555,1.111778,0,0
1,success,1528,1528,0,500,500,500,1028,0.327225,1,...,0,0.0,0,1028,1.0,0.229448,0.559,1.154788,1,1
2,success,1546,1546,0,499,499,499,1047,0.322768,1,...,0,0.0,0,1047,1.0,0.220475,0.561,1.099218,2,2
3,success,1502,1502,0,470,470,470,1032,0.312916,1,...,0,0.0,0,1032,1.0,0.211834,0.570,1.130144,3,3
4,success,1527,1527,0,516,516,516,1011,0.337917,1,...,0,0.0,0,1011,1.0,0.215063,0.583,1.090944,4,4


===== FRAME STATUS SUMMARY =====


,count,count
0,success,66


In [40]:
# Cell 6 - Summary file dan frame yang perlu dicek

print("===== FILE-LEVEL SUMMARY =====")

summary = {
    "file_path": str(CHECK_FILE_PATH),
    "n_frames": len(frame_metrics_df),
    "eps": EPS,
    "min_samples": MIN_SAMPLES,
    "min_cluster_points": MIN_CLUSTER_POINTS,

    "n_points_input_mean": frame_metrics_df["n_points_input"].mean(),
    "n_points_after_cleaning_mean": frame_metrics_df["n_points_after_cleaning"].mean(),
    "cleaning_removed_ratio_mean": frame_metrics_df["cleaning_removed_ratio"].mean(),

    "n_corr_all_zero_removed_mean": frame_metrics_df["n_corr_all_zero_removed"].mean(),
    "n_raw_all_zero_removed_mean": frame_metrics_df["n_raw_all_zero_removed"].mean(),
    "n_all_zero_removed_mean": frame_metrics_df["n_all_zero_removed"].mean(),

    "noise_ratio_mean": frame_metrics_df["noise_ratio"].mean(),
    "cluster_count_valid_mean": frame_metrics_df["cluster_count_valid"].mean(),
    "main_cluster_ratio_total_mean": frame_metrics_df["main_cluster_ratio_total"].mean(),
    "main_cluster_points_mean": frame_metrics_df["main_cluster_points"].mean(),
}

summary_df = pd.DataFrame([summary])
display(summary_df.T.rename(columns={0: "value"}))

print("===== POTENTIALLY DIFFICULT FRAMES =====")

difficult_frames = frame_metrics_df.sort_values(
    by=["main_cluster_ratio_total", "noise_ratio"],
    ascending=[True, False],
).head(10)

display(
    difficult_frames[
        [
            "frame_id",
            "status",
            "n_points_input",
            "n_points_after_cleaning",
            "cleaning_removed_ratio",
            "cluster_count_total",
            "cluster_count_valid",
            "noise_ratio",
            "main_cluster_points",
            "main_cluster_ratio_total",
            "main_cluster_x_range",
            "main_cluster_y_range",
            "main_cluster_z_range",
        ]
    ]
)

===== FILE-LEVEL SUMMARY =====


,value
file_path,/media/spell/Spell-lab/Lidar/Preprocessing/Pre...
n_frames,66
eps,0.2
min_samples,15
min_cluster_points,20
n_points_input_mean,1779.969697
n_points_after_cleaning_mean,1099.060606
cleaning_removed_ratio_mean,0.376737
n_corr_all_zero_removed_mean,680.909091
n_raw_all_zero_removed_mean,680.909091


===== POTENTIALLY DIFFICULT FRAMES =====


,frame_id,status,n_points_input,n_points_after_cleaning,cleaning_removed_ratio,cluster_count_total,cluster_count_valid,noise_ratio,main_cluster_points,main_cluster_ratio_total,main_cluster_x_range,main_cluster_y_range,main_cluster_z_range
33,33,success,1374,885,0.355895,2,2,0.000000,768,0.867797,0.728449,0.478,0.839852
64,64,success,2080,999,0.519712,1,1,0.001001,998,0.998999,0.717417,0.484,0.754203
0,0,success,1478,973,0.341678,1,1,0.000000,973,1.000000,0.202299,0.555,1.111778
1,1,success,1528,1028,0.327225,1,1,0.000000,1028,1.000000,0.229448,0.559,1.154788
2,2,success,1546,1047,0.322768,1,1,0.000000,1047,1.000000,0.220475,0.561,1.099218
3,3,success,1502,1032,0.312916,1,1,0.000000,1032,1.000000,0.211834,0.570,1.130144
4,4,success,1527,1011,0.337917,1,1,0.000000,1011,1.000000,0.215063,0.583,1.090944
5,5,success,1527,1008,0.339882,1,1,0.000000,1008,1.000000,0.221789,0.579,1.120907
6,6,success,1529,1011,0.338784,1,1,0.000000,1011,1.000000,0.220018,0.559,1.184386
7,7,success,1518,1013,0.332675,1,1,0.000000,1013,1.000000,0.218310,0.568,1.123378


In [41]:
# Cell 7 - Fungsi visualisasi 3D berdasarkan mode

def get_plot_dataframe(labeled_df: pd.DataFrame, mode: str):
    """
    Mengambil dataframe sesuai mode visualisasi.
    """
    if labeled_df is None or len(labeled_df) == 0:
        return pd.DataFrame()

    if mode == "labels":
        plot_df = labeled_df.copy()
        color_values = plot_df["dbscan_label"]
        color_title = "DBSCAN label"
        title_suffix = "All cleaned points colored by DBSCAN label"

    elif mode == "main_only":
        plot_df = labeled_df[labeled_df["is_main_cluster"]].copy()
        color_values = plot_df["Z_level"] if len(plot_df) > 0 else []
        color_title = "Z_level"
        title_suffix = "Main cluster only"

    elif mode == "noise_vs_cluster":
        plot_df = labeled_df.copy()
        plot_df["noise_vs_cluster"] = np.where(plot_df["dbscan_label"] == -1, 0, 1)
        color_values = plot_df["noise_vs_cluster"]
        color_title = "0=noise, 1=cluster"
        title_suffix = "Noise vs non-noise cluster"

    elif mode == "valid_clusters_only":
        plot_df = labeled_df[labeled_df["is_valid_cluster"]].copy()
        color_values = plot_df["dbscan_label"] if len(plot_df) > 0 else []
        color_title = "Valid cluster label"
        title_suffix = "Valid clusters only"

    elif mode == "raw_cleaned":
        plot_df = labeled_df.copy()
        color_values = plot_df["Z_level"]
        color_title = "Z_level"
        title_suffix = "All cleaned points colored by Z_level"

    else:
        raise ValueError(f"Unknown visual mode: {mode}")

    return plot_df, color_values, color_title, title_suffix


def make_frame_figure(frame_id, mode):
    """
    Membuat plotly figure untuk satu frame dan satu mode.
    """
    result = frame_results[frame_id]
    labeled_df = result["labeled_df"]
    metrics = result["metrics"]

    plot_df, color_values, color_title, title_suffix = get_plot_dataframe(labeled_df, mode)

    if len(plot_df) > MAX_POINTS_PLOT:
        plot_df = plot_df.sample(MAX_POINTS_PLOT, random_state=42)
        if isinstance(color_values, pd.Series):
            color_values = plot_df[color_values.name]
        elif isinstance(color_values, np.ndarray):
            color_values = color_values[:len(plot_df)]

    fig = go.Figure()

    if len(plot_df) > 0:
        fig.add_trace(
            go.Scatter3d(
                x=plot_df["X_corr"],
                y=plot_df["Y_corr"],
                z=plot_df["Z_level"],
                mode="markers",
                marker=dict(
                    size=2,
                    color=color_values,
                    colorscale="Turbo",
                    opacity=0.80,
                    colorbar=dict(title=color_title),
                ),
                text=[
                    f"label={lab}<br>"
                    f"X={x:.3f}<br>Y={y:.3f}<br>Z={z:.3f}"
                    for lab, x, y, z in zip(
                        plot_df["dbscan_label"],
                        plot_df["X_corr"],
                        plot_df["Y_corr"],
                        plot_df["Z_level"],
                    )
                ],
                hoverinfo="text",
                name="points",
            )
        )

    fig.update_layout(
        title=(
            f"DBSCAN Visual Validation<br>"
            f"Frame: {frame_id} | Mode: {mode} | {title_suffix}<br>"
            f"eps={EPS}, min_samples={MIN_SAMPLES}, min_cluster_points={MIN_CLUSTER_POINTS}"
        ),
        scene=dict(
            xaxis=dict(title="X_corr", range=[ROI_X_MIN, ROI_X_MAX]),
            yaxis=dict(title="Y_corr", range=[ROI_Y_MIN, ROI_Y_MAX]),
            zaxis=dict(title="Z_level", range=[ROI_Z_MIN, ROI_Z_MAX]),
            aspectmode="manual",
            aspectratio=dict(x=1.5, y=1.5, z=1.0),
        ),
        width=950,
        height=750,
        margin=dict(l=0, r=0, b=0, t=110),
    )

    return fig

In [42]:
# Cell 8 - Fungsi summary GUI per frame

def make_summary_html(frame_id):
    result = frame_results[frame_id]
    metrics = result["metrics"]

    rows = [
        ("frame_id", frame_id),
        ("status", metrics.get("status")),
        ("n_points_input", metrics.get("n_points_input")),
        ("n_points_before_cleaning", metrics.get("n_points_before_cleaning")),
        ("n_corr_all_zero_removed", metrics.get("n_corr_all_zero_removed")),
        ("n_raw_all_zero_removed", metrics.get("n_raw_all_zero_removed")),
        ("n_all_zero_removed", metrics.get("n_all_zero_removed")),
        ("n_points_after_cleaning", metrics.get("n_points_after_cleaning")),
        ("cleaning_removed_ratio", metrics.get("cleaning_removed_ratio")),
        ("cluster_count_total", metrics.get("cluster_count_total")),
        ("cluster_count_valid", metrics.get("cluster_count_valid")),
        ("noise_points", metrics.get("noise_points")),
        ("noise_ratio", metrics.get("noise_ratio")),
        ("main_cluster_label", metrics.get("main_cluster_label")),
        ("main_cluster_points", metrics.get("main_cluster_points")),
        ("main_cluster_ratio_total", metrics.get("main_cluster_ratio_total")),
        ("main_cluster_x_range", metrics.get("main_cluster_x_range")),
        ("main_cluster_y_range", metrics.get("main_cluster_y_range")),
        ("main_cluster_z_range", metrics.get("main_cluster_z_range")),
    ]

    html = """
    <div style="font-family: Arial; font-size: 13px;">
    <h3>Frame Summary</h3>
    <table style="border-collapse: collapse;">
    """

    for k, v in rows:
        if isinstance(v, float):
            if np.isnan(v):
                v_str = "NaN"
            else:
                v_str = f"{v:.6f}"
        else:
            v_str = str(v)

        html += f"""
        <tr>
            <td style="border: 1px solid #ccc; padding: 4px 8px;"><b>{k}</b></td>
            <td style="border: 1px solid #ccc; padding: 4px 8px;">{v_str}</td>
        </tr>
        """

    html += """
    </table>
    </div>
    """

    return HTML(html)

In [43]:
# Cell 9 - GUI interaktif dengan slider, Play, Prev, Next, dan mode

if len(frame_ids) == 0:
    raise ValueError("No frames available for visualization.")

frame_index_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    description="Frame idx:",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)

play_widget = widgets.Play(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    interval=500,
    description="Play",
    disabled=False,
)

widgets.jslink((play_widget, "value"), (frame_index_slider, "value"))

mode_dropdown = widgets.Dropdown(
    options=VISUAL_MODES,
    value=DEFAULT_VISUAL_MODE,
    description="Mode:",
    layout=widgets.Layout(width="320px"),
)

prev_button = widgets.Button(
    description="Prev Frame",
    button_style="",
    tooltip="Go to previous frame",
    icon="arrow-left",
)

next_button = widgets.Button(
    description="Next Frame",
    button_style="",
    tooltip="Go to next frame",
    icon="arrow-right",
)

frame_label = widgets.HTML()

plot_output = widgets.Output()
summary_output = widgets.Output()


def update_frame_label():
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]
    frame_label.value = (
        f"<b>Current:</b> index {idx + 1}/{len(frame_ids)} | "
        f"<b>frame_id:</b> {frame_id}"
    )


def render_current_frame(change=None):
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]
    mode = mode_dropdown.value

    update_frame_label()

    with plot_output:
        clear_output(wait=True)
        fig = make_frame_figure(frame_id, mode)
        fig.show()

    with summary_output:
        clear_output(wait=True)
        display(make_summary_html(frame_id))


def on_prev_clicked(b):
    current = frame_index_slider.value
    if current > frame_index_slider.min:
        frame_index_slider.value = current - 1


def on_next_clicked(b):
    current = frame_index_slider.value
    if current < frame_index_slider.max:
        frame_index_slider.value = current + 1


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

frame_index_slider.observe(render_current_frame, names="value")
mode_dropdown.observe(render_current_frame, names="value")

controls = widgets.VBox([
    widgets.HBox([play_widget, frame_index_slider]),
    widgets.HBox([prev_button, next_button, mode_dropdown]),
    frame_label,
])

display(controls)
display(summary_output)
display(plot_output)

render_current_frame()

Output()

Output()

In [44]:
# Cell 10 - Quick filter: tampilkan frame bermasalah untuk dipilih manual

print("===== LOWEST MAIN CLUSTER RATIO FRAMES =====")

cols = [
    "frame_index",
    "frame_id",
    "status",
    "n_points_input",
    "n_points_after_cleaning",
    "cleaning_removed_ratio",
    "cluster_count_total",
    "cluster_count_valid",
    "noise_ratio",
    "main_cluster_points",
    "main_cluster_ratio_total",
    "main_cluster_x_range",
    "main_cluster_y_range",
    "main_cluster_z_range",
]

display(
    frame_metrics_df[cols]
    .sort_values(["main_cluster_ratio_total", "noise_ratio"], ascending=[True, False])
    .head(15)
)

print("===== HIGHEST NOISE RATIO FRAMES =====")

display(
    frame_metrics_df[cols]
    .sort_values(["noise_ratio", "main_cluster_ratio_total"], ascending=[False, True])
    .head(15)
)

===== LOWEST MAIN CLUSTER RATIO FRAMES =====


,frame_index,frame_id,status,n_points_input,n_points_after_cleaning,cleaning_removed_ratio,cluster_count_total,cluster_count_valid,noise_ratio,main_cluster_points,main_cluster_ratio_total,main_cluster_x_range,main_cluster_y_range,main_cluster_z_range
33,33,33,success,1374,885,0.355895,2,2,0.000000,768,0.867797,0.728449,0.478,0.839852
64,64,64,success,2080,999,0.519712,1,1,0.001001,998,0.998999,0.717417,0.484,0.754203
0,0,0,success,1478,973,0.341678,1,1,0.000000,973,1.000000,0.202299,0.555,1.111778
1,1,1,success,1528,1028,0.327225,1,1,0.000000,1028,1.000000,0.229448,0.559,1.154788
2,2,2,success,1546,1047,0.322768,1,1,0.000000,1047,1.000000,0.220475,0.561,1.099218
3,3,3,success,1502,1032,0.312916,1,1,0.000000,1032,1.000000,0.211834,0.570,1.130144
4,4,4,success,1527,1011,0.337917,1,1,0.000000,1011,1.000000,0.215063,0.583,1.090944
5,5,5,success,1527,1008,0.339882,1,1,0.000000,1008,1.000000,0.221789,0.579,1.120907
6,6,6,success,1529,1011,0.338784,1,1,0.000000,1011,1.000000,0.220018,0.559,1.184386
7,7,7,success,1518,1013,0.332675,1,1,0.000000,1013,1.000000,0.218310,0.568,1.123378


===== HIGHEST NOISE RATIO FRAMES =====


,frame_index,frame_id,status,n_points_input,n_points_after_cleaning,cleaning_removed_ratio,cluster_count_total,cluster_count_valid,noise_ratio,main_cluster_points,main_cluster_ratio_total,main_cluster_x_range,main_cluster_y_range,main_cluster_z_range
64,64,64,success,2080,999,0.519712,1,1,0.001001,998,0.998999,0.717417,0.484,0.754203
33,33,33,success,1374,885,0.355895,2,2,0.000000,768,0.867797,0.728449,0.478,0.839852
0,0,0,success,1478,973,0.341678,1,1,0.000000,973,1.000000,0.202299,0.555,1.111778
1,1,1,success,1528,1028,0.327225,1,1,0.000000,1028,1.000000,0.229448,0.559,1.154788
2,2,2,success,1546,1047,0.322768,1,1,0.000000,1047,1.000000,0.220475,0.561,1.099218
3,3,3,success,1502,1032,0.312916,1,1,0.000000,1032,1.000000,0.211834,0.570,1.130144
4,4,4,success,1527,1011,0.337917,1,1,0.000000,1011,1.000000,0.215063,0.583,1.090944
5,5,5,success,1527,1008,0.339882,1,1,0.000000,1008,1.000000,0.221789,0.579,1.120907
6,6,6,success,1529,1011,0.338784,1,1,0.000000,1011,1.000000,0.220018,0.559,1.184386
7,7,7,success,1518,1013,0.332675,1,1,0.000000,1013,1.000000,0.218310,0.568,1.123378


In [45]:
# Cell 11 - Jump ke frame tertentu berdasarkan frame_id

def jump_to_frame_id(target_frame_id):
    """
    Panggil fungsi ini untuk lompat ke frame_id tertentu.
    Contoh:
    jump_to_frame_id(31)
    """
    if target_frame_id not in frame_ids:
        print(f"Frame ID {target_frame_id} not found.")
        print(f"Available range: {frame_ids[0]} to {frame_ids[-1]}")
        return

    idx = frame_ids.index(target_frame_id)
    frame_index_slider.value = idx
    print(f"Jumped to frame_id={target_frame_id}, index={idx}")

# Contoh penggunaan:
# jump_to_frame_id(31)